# Help Twitter Combat Hate Speech
Description: Using NLP and ML, build a model to identify hate speech (racist or sexist tweets) on Twitter.

## 1. Import libraries and load data

In [38]:
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.tokenize import TweetTokenizer
import nltk
from collections import Counter
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, f1_score

In [39]:
# Download stopwords
nltk.download('stopwords')

# Load data
tweets_df = pd.read_csv("TwitterHate.csv")
tweets = tweets_df['tweet'].tolist()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Vignesh_S16\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 2. Text cleanup

In [40]:
stop_words = set(stopwords.words('english'))
tokenizer = TweetTokenizer()
cleaned_tweets = []

for tweet in tweets:
    tweet = tweet.lower()
    tweet = re.sub(r'@\w+', '', tweet)
    tweet = re.sub(r'http\S+|www\S+|https\S+', '', tweet)

    tokens = tokenizer.tokenize(tweet)
    tokens = [t for t in tokens if t not in stop_words and t not in ['amp', 'rt']]

    tokens = [t.replace('#','') for t in tokens]

    tokens = [t for t in tokens if len(t) > 1]
    cleaned_tweets.append(tokens)

## 3. Explore top used terms

In [57]:
all_terms = [term for tokens in cleaned_tweets for term in tokens]
top_terms = Counter(all_terms).most_common(10)

df_top_terms = pd.DataFrame(top_terms, columns=['Term', 'Frequency'])
df_top_terms

,Term,Frequency
0,...,2808
1,love,2748
2,day,2276
3,happy,1684
4,time,1131
5,life,1118
6,like,1047
7,today,1013
8,new,994
9,thankful,946


## 4. Prepare data for modeling

In [42]:
cleaned_tweets_str = [" ".join(tokens) for tokens in cleaned_tweets]
X = cleaned_tweets_str
y = tweets_df['label']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## 5. TF-IDF vectorization

In [43]:
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

## 6. Logistic Regression - default

In [44]:
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

LogisticRegression()

## 7. Evaluate default model

In [45]:
y_train_pred = lr.predict(X_train_tfidf)
print("Default Logistic Regression - Train Accuracy:", accuracy_score(y_train, y_train_pred))
print("Default Logistic Regression - Train Recall:", recall_score(y_train, y_train_pred))
print("Default Logistic Regression - Train F1 Score:", f1_score(y_train, y_train_pred))

Default Logistic Regression - Train Accuracy: 0.955844968516563
Default Logistic Regression - Train Recall: 0.39409141583054624
Default Logistic Regression - Train F1 Score: 0.5560361777428234


## 8. Adjust class imbalance

In [46]:
lr_balanced = LogisticRegression(class_weight='balanced')
lr_balanced.fit(X_train_tfidf, y_train)

LogisticRegression(class_weight='balanced')

## 9. Evaluate balanced model

In [47]:
y_train_pred_bal = lr_balanced.predict(X_train_tfidf)
print("Balanced Logistic Regression - Train Accuracy:", accuracy_score(y_train, y_train_pred_bal))
print("Balanced Logistic Regression - Train Recall:", recall_score(y_train, y_train_pred_bal))
print("Balanced Logistic Regression - Train F1 Score:", f1_score(y_train, y_train_pred_bal))

Balanced Logistic Regression - Train Accuracy: 0.9467323712307872
Balanced Logistic Regression - Train Recall: 0.9671125975473801
Balanced Logistic Regression - Train F1 Score: 0.7181291390728477


## 10. Hyperparameter tuning with GridSearchCV

In [48]:
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']  # Required for l1 penalty
}

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
grid = GridSearchCV(
    LogisticRegression(class_weight='balanced'), 
    param_grid, 
    scoring='recall', 
    cv=cv
)
grid.fit(X_train_tfidf, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=4, random_state=42, shuffle=True),
             estimator=LogisticRegression(class_weight='balanced'),
             param_grid={'C': [0.01, 0.1, 1, 10], 'penalty': ['l1', 'l2'],
                         'solver': ['liblinear']},
             scoring='recall')

## 11. Best parameters

In [49]:
print("Best Parameters:", grid.best_params_)

Best Parameters: {'C': 1, 'penalty': 'l2', 'solver': 'liblinear'}


## 12. Evaluate best estimator on test set

In [50]:
best_lr = grid.best_estimator_
y_test_pred = best_lr.predict(X_test_tfidf)
print("Test Recall (toxic comments):", recall_score(y_test, y_test_pred))
print("Test F1 Score:", f1_score(y_test, y_test_pred))

Test Recall (toxic comments): 0.7879464285714286
Test F1 Score: 0.5937762825904122


In [60]:
print("\nBalanced Logistic Regression - Train Metrics")
print(f"Accuracy: {accuracy_score(y_train, y_train_pred_bal):.4f}")
print(f"Recall: {recall_score(y_train, y_train_pred_bal):.4f}")
print(f"F1 Score: {f1_score(y_train, y_train_pred_bal):.4f}")

print("\nBest Estimator - Test Metrics")
print(f"Recall (toxic comments): {recall_score(y_test, y_test_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_test_pred):.4f}")



Balanced Logistic Regression - Train Metrics
Accuracy: 0.9467
Recall: 0.9671
F1 Score: 0.7181

Best Estimator - Test Metrics
Recall (toxic comments): 0.7879
F1 Score: 0.5938


In [54]:
def predict_tweet(tweet_text):
    tweet = tweet_text.lower()
    tweet = re.sub(r'@\w+', '', tweet)
    tweet = re.sub(r'http\S+|www\S+|https\S+', '', tweet)
    tokens = tokenizer.tokenize(tweet)
    tokens = [t for t in tokens if t not in stop_words and t not in ['amp', 'rt']]
    tokens = [t.replace('#','') for t in tokens if len(t) > 1]
    tweet_cleaned = " ".join(tokens)
    tweet_vector = tfidf.transform([tweet_cleaned])
    pred = best_lr.predict(tweet_vector)[0]
    return "HATE SPEECH" if pred == 1 else "NON-HATE"


In [55]:
print(predict_tweet("I love sunny days!"))
print(predict_tweet("I hate everyone!"))

NON-HATE
HATE SPEECH
